# PASTA — Colab setup: checkpoints + data → Google Drive

End-to-end setup for **[PASTA (ICCV 2025)](https://github.com/kuai-lab/iccv25_pasta)** — *Part-Aware Sketch-to-3D Shape Generation with Text-Aligned Prior* — from the fork **[SattamAltwaim/iccv25_pasta](https://github.com/SattamAltwaim/iccv25_pasta)**.

What this notebook does:

| Step | Artifact | Size | Where it persists |
|---|---|---|---|
| 1 | PASTA pretrained sketch model (`sketch2spaghetti_chairs/model`) | **8.4 GB** | `Drive/PASTA/checkpoints/` |
| 2 | SPAGHETTI backbone (`occ_gmm_chairs_sym_hard`, from SENS) | ~0.1–1 GB | `Drive/PASTA/checkpoints/` |
| 3 | LLaVA-7B features for AmateurSketch-3D eval | 0.56 GB zip | `Drive/PASTA/archives/` |
| 4 | SENS preprocessed chair training set (`dataset_chair_preprocess`) | several GB zip | `Drive/PASTA/archives/` |
| 5 | Smoke test: `run.py --input sketch.png` → mesh | – | `Drive/PASTA/outputs/` |

**Requirements**
- GPU runtime (`Runtime → Change runtime type → T4 GPU`). Free tier works; High-RAM is more comfortable (the PASTA checkpoint bundles optimizer state and is loaded to CPU RAM first).
- ≥ **13 GB free Google Drive space** with all downloads enabled (flip the flags below to skip pieces).
- Optional: a Colab secret `GITHUB_TOKEN` (repo-scoped PAT, set via the 🔑 sidebar) so the notebook can **push run results back to GitHub** — that's how Colab runs show up when you `git pull` locally.

**Design notes**
- Every download is **idempotent and resumable** — re-running any cell skips what's already on Drive.
- Big archives live on Drive; datasets with tens of thousands of small files are extracted to fast **local** session storage each run (Drive mounts choke on many small files). Run the *Session restore* section at the start of every new session.
- The evaluation sketches + ShapeNet meshes (`shapenet_amateur`) are **license-gated** and cannot be auto-downloaded — see the Evaluation section for exact instructions and expected layout.


## 0 · Runtime check

In [ ]:
import torch, subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout or out.stderr)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU, then restart.'
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

## 1 · Configuration

All flags in one place. Defaults download everything needed to later finetune on Fusion360.

In [ ]:
import os, shutil
from pathlib import Path

GITHUB_USER = 'SattamAltwaim'
REPO_NAME   = 'iccv25_pasta'
REPO_DIR    = Path('/content') / REPO_NAME

DRIVE_ROOT  = Path('/content/drive/MyDrive/PASTA')  # everything persistent lives under here

# --- what to set up (all steps idempotent; re-running skips finished work) ---
DOWNLOAD_PASTA_CHECKPOINT   = True   # 8.4 GB -> checkpoints/sketch2spaghetti_chairs/model
DOWNLOAD_SPAGHETTI_BACKBONE = True   # SENS pretrained zip -> checkpoints/occ_gmm_chairs_sym_hard
DOWNLOAD_LLAVA_FEATURES     = True   # 0.56 GB zip (AmateurSketch eval features)
DOWNLOAD_TRAIN_DATASET      = True   # SENS dataset_chair_preprocess (needed for training/finetuning)
COMPUTE_MU_DISTANCES        = True   # rebuild training adjacency files the authors did not release
RUN_SMOKE_TEST              = True   # run.py on the bundled sketch.png — verifies the full pipeline
RUN_EVAL                    = False  # needs the manual, license-gated shapenet_amateur set (see below)
PUSH_RUN_RESULTS            = True   # commit small artifacts to GitHub under colab_runs/

# --- sources ---
PASTA_MODEL_URL  = 'https://kuaicv.synology.me/weights/iccv2025/PASTA/model'
PASTA_MODEL_SIZE = 9001546341  # bytes, integrity check (from the server's Content-Length)
LLAVA_ZIP_URL    = 'https://kuaicv.synology.me/weights/iccv2025/PASTA/shapenet_amateur.zip'
# Google Drive ids published in the SENS README (github.com/AlexandreBinninger/SENS):
SENS_CHECKPOINTS_GDRIVE_ID = '15b98DQL3ebL_SaccRys0zJimCxppxaVo'
SENS_DATASET_GDRIVE_ID     = '10YHyS2O-SPwKFBR65aZE0KEpvaUQVfzA'
print('config ready')

## 2 · Mount Google Drive & create the layout

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR    = DRIVE_ROOT / 'checkpoints'
ARCHIVE_DIR = DRIVE_ROOT / 'archives'
MANUAL_DIR  = ARCHIVE_DIR / 'manual'    # drop license-gated zips here yourself
OUT_DIR     = DRIVE_ROOT / 'outputs'
GEN_DIR     = DRIVE_ROOT / 'generated'  # things we compute (e.g. mu-distance adjacency)
for d in (CKPT_DIR, ARCHIVE_DIR, MANUAL_DIR, OUT_DIR, GEN_DIR):
    d.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage('/content/drive')
print(f'Drive free space: {free/1e9:.1f} GB (need ~13 GB for a full setup)')
print('layout ready under', DRIVE_ROOT)

## 3 · Clone the repo

Reads the Colab secret `GITHUB_TOKEN` if you configured one (🔑 sidebar → add secret, grant notebook access). With a token, the final cell can push run results back to GitHub; without one everything still works read-only.

In [ ]:
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    print('no GITHUB_TOKEN secret — cloning read-only (pushing results will be disabled)')

_auth = f'{token}@' if token else ''
clone_url = f'https://{_auth}github.com/{GITHUB_USER}/{REPO_NAME}.git'

if (REPO_DIR / '.git').exists():
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone "{clone_url}" {REPO_DIR} 2>&1 | grep -v http   # avoid echoing the token
%cd {REPO_DIR}
!git log --oneline -3

## 4 · Install dependencies

Colab already ships torch/torchvision/OpenCV/scipy — only the missing pieces are installed (no torch reinstall, so this stays fast and CUDA-safe).

In [ ]:
%pip install -q trimesh point-cloud-utils libigl cairosvg lxml plyfile rtree

import trimesh, igl, cairosvg, plyfile
import point_cloud_utils as pcu
print('dependency imports OK')

## 5 · Downloads → Google Drive

Helper functions first; each following cell checks Drive before downloading, so re-runs are free and interrupted downloads resume (`wget -c`).

In [ ]:
import zipfile, tarfile

def dl_resumable(url, dest: Path, expected_size=None):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and (expected_size is None or dest.stat().st_size == expected_size):
        print(f'[skip] {dest.name} already on Drive ({dest.stat().st_size/1e9:.2f} GB)')
        return dest
    print(f'[download] {url}\n        -> {dest}')
    ret = os.system(f'wget -c -q --show-progress -O "{dest}" "{url}"')
    assert ret == 0, f'download failed — re-run this cell to resume: {url}'
    if expected_size is not None:
        got = dest.stat().st_size
        assert got == expected_size, f'size mismatch {got} != {expected_size} — re-run this cell to resume'
    return dest

def gdrive_download(file_id, dest: Path):
    if dest.exists() and dest.stat().st_size > 0:
        print(f'[skip] {dest.name} already on Drive ({dest.stat().st_size/1e9:.2f} GB)')
        return dest
    import gdown
    dest.parent.mkdir(parents=True, exist_ok=True)
    out = gdown.download(id=file_id, output=str(dest), quiet=False)
    assert out is not None, (
        f'gdown failed (often a shared-file download quota). Open '
        f'https://drive.google.com/file/d/{file_id}/view in your browser, save a copy to your '
        f'own Drive, and place/rename it at {dest} — then re-run.')
    return dest

def extract_archive(archive: Path, dest: Path, members_containing=None):
    dest.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            names = z.namelist()
            if members_containing:
                names = [n for n in names if members_containing in n]
            z.extractall(dest, members=names)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as t:
            members = t.getmembers()
            if members_containing:
                members = [m for m in members if members_containing in m.name]
            t.extractall(dest, members=members)
    else:
        raise RuntimeError(f'unrecognized archive format: {archive}')
    return dest

def find_dir_named(root: Path, name: str):
    if (root / name).is_dir():
        return root / name
    return next((p for p in root.rglob(name) if p.is_dir()), None)

print('helpers ready')

### 5.1 · PASTA pretrained checkpoint (8.4 GB — expect 5–30 min)

Goes to `Drive/PASTA/checkpoints/sketch2spaghetti_chairs/model`, which is exactly where `run.py` (tag `chairs`) and `eval.py --tag chairs` look for it via the `assets/checkpoints` symlink.

In [ ]:
pasta_ckpt = CKPT_DIR / 'sketch2spaghetti_chairs' / 'model'
if DOWNLOAD_PASTA_CHECKPOINT:
    dl_resumable(PASTA_MODEL_URL, pasta_ckpt, PASTA_MODEL_SIZE)

### 5.2 · SPAGHETTI backbone (`occ_gmm_chairs_sym_hard`)

PASTA decodes shapes through a frozen SPAGHETTI model; the released weights come from the SENS pretrained bundle. **Only** the `occ_gmm_chairs_sym_hard` folder is taken — the bundle's own `sketch2spaghetti_chairs` is SENS's model, architecturally different from PASTA's, and must not overwrite the 8.4 GB PASTA checkpoint.

In [ ]:
backbone_dir = CKPT_DIR / 'occ_gmm_chairs_sym_hard'
if DOWNLOAD_SPAGHETTI_BACKBONE and not (backbone_dir / 'model').exists():
    sens_zip = gdrive_download(SENS_CHECKPOINTS_GDRIVE_ID, Path('/content/sens_checkpoints.zip'))
    tmp = Path('/content/sens_ckpt_extract'); shutil.rmtree(tmp, ignore_errors=True)
    extract_archive(sens_zip, tmp, members_containing='occ_gmm_chairs_sym_hard')
    src = find_dir_named(tmp, 'occ_gmm_chairs_sym_hard')
    assert src is not None, 'occ_gmm_chairs_sym_hard not found inside the SENS bundle — inspect /content/sens_checkpoints.zip'
    shutil.copytree(src, backbone_dir, dirs_exist_ok=True)
    shutil.rmtree(tmp, ignore_errors=True)
print('backbone files:', sorted(p.name for p in backbone_dir.glob('*')) if backbone_dir.exists() else 'MISSING')

### 5.3 · LLaVA-7B features for AmateurSketch-3D (0.56 GB)

Used by `eval.py` (`l_feat` input). The zip stays on Drive; it is unpacked to local storage in the *Session restore* step.

In [ ]:
llava_zip = ARCHIVE_DIR / 'shapenet_amateur_llava.zip'
if DOWNLOAD_LLAVA_FEATURES:
    dl_resumable(LLAVA_ZIP_URL, llava_zip)

### 5.4 · SENS preprocessed chair training set (several GB)

`dataset_chair_preprocess` — required for training and the reference layout you'll mirror when preparing **Fusion360**. The zip stays on Drive; extraction happens locally each session (it contains tens of thousands of small files, which Drive handles poorly).

In [ ]:
dataset_zip = ARCHIVE_DIR / 'dataset_chair_preprocess.zip'
if DOWNLOAD_TRAIN_DATASET:
    gdrive_download(SENS_DATASET_GDRIVE_ID, dataset_zip)

## 6 · Session restore — run at the start of **every** session

Wires the repo's `assets/` to Drive (checkpoints, outputs) and unpacks the archives from Drive into fast local storage (data). Everything is derived from Drive, so a fresh VM gets back to a working state in a few minutes without re-downloading.

In [ ]:
LOCAL_DATA = Path('/content/pasta_data')
LOCAL_DATA.mkdir(exist_ok=True)
assets = REPO_DIR / 'assets'
assets.mkdir(exist_ok=True)

def relink(link: Path, target: Path):
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)  # placeholder dirs from the repo; real data lives on Drive/local
    link.symlink_to(target)
    print(f'{link.relative_to(REPO_DIR)} -> {target}')

relink(assets / 'checkpoints', CKPT_DIR)
relink(assets / 'data', LOCAL_DATA)
relink(assets / 'output', OUT_DIR)

In [ ]:
# LLaVA eval features -> assets/data/chair_llava_feat/shapenet_amateur/<obj_id>/<view>.npy
llava_target = LOCAL_DATA / 'chair_llava_feat' / 'shapenet_amateur'
if llava_zip.exists() and not llava_target.exists():
    tmp = Path('/content/tmp_llava'); shutil.rmtree(tmp, ignore_errors=True)
    extract_archive(llava_zip, tmp)
    some_npy = next(tmp.rglob('*.npy'), None)
    assert some_npy is not None, 'no .npy files inside the LLaVA archive?'
    root = some_npy.parent.parent  # layout is <root>/<obj_id>/<view>.npy
    llava_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(root), str(llava_target))
    shutil.rmtree(tmp, ignore_errors=True)
print('llava feature files:', len(list(llava_target.rglob('*.npy'))) if llava_target.exists() else 'MISSING')

In [ ]:
# Training set -> assets/data/dataset_chair_preprocess (local; a few minutes to unpack)
dataset_target = LOCAL_DATA / 'dataset_chair_preprocess'
if dataset_zip.exists() and not dataset_target.exists():
    tmp = Path('/content/tmp_dataset'); shutil.rmtree(tmp, ignore_errors=True)
    print('extracting training dataset locally (many small files, be patient)...')
    extract_archive(dataset_zip, tmp)
    src = find_dir_named(tmp, 'dataset_chair_preprocess') or tmp
    shutil.move(str(src), str(dataset_target))
    shutil.rmtree(tmp, ignore_errors=True)

# restore previously generated training prereqs from Drive
if dataset_target.exists():
    for f in GEN_DIR.glob('chairs_mu_distances*.npy'):
        if not (dataset_target / f.name).exists():
            shutil.copy2(f, dataset_target / f.name)
            print('restored from Drive:', f.name)
print('training dataset:', 'ready' if dataset_target.exists() else 'not downloaded')

In [ ]:
# Optional manual eval set (license-gated, see the Evaluation section):
# put a zip at Drive/PASTA/archives/manual/shapenet_amateur.zip with layout <obj_id>/{1.png,2.png,3.png,model.obj}
manual_zip = MANUAL_DIR / 'shapenet_amateur.zip'
eval_target = LOCAL_DATA / 'shapenet_amateur'
if manual_zip.exists() and not eval_target.exists():
    tmp = Path('/content/tmp_eval'); shutil.rmtree(tmp, ignore_errors=True)
    extract_archive(manual_zip, tmp)
    src = find_dir_named(tmp, 'shapenet_amateur') or tmp
    shutil.move(str(src), str(eval_target))
    shutil.rmtree(tmp, ignore_errors=True)
print('eval set:', 'ready' if eval_target.exists() else 'not present (only needed for eval.py)')

## 7 · Verify everything

In [ ]:
def _size_gb(p: Path):
    if p.is_file():
        return p.stat().st_size / 1e9
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e9

checks = {
    'PASTA sketch model   checkpoints/sketch2spaghetti_chairs/model': pasta_ckpt,
    'SPAGHETTI backbone   checkpoints/occ_gmm_chairs_sym_hard/model': backbone_dir / 'model',
    'SPAGHETTI options    checkpoints/occ_gmm_chairs_sym_hard/options.pkl': backbone_dir / 'options.pkl',
    'LLaVA eval features  data/chair_llava_feat/shapenet_amateur': llava_target,
    'Training dataset     data/dataset_chair_preprocess': dataset_target,
    'Eval set (manual)    data/shapenet_amateur': eval_target,
}
all_ok = True
for name, p in checks.items():
    if p.exists():
        print(f'  OK   {name}  ({_size_gb(p):.2f} GB)')
    else:
        print(f'  MISS {name}')
        all_ok = all_ok and 'manual' in name  # the manual eval set is allowed to be absent
print('\nready for inference' if all_ok else '\nsome required pieces are missing — check the cells above')

## 8 · Training prerequisite: GMM adjacency matrices

The trainers load `chairs_mu_distances.npy` / `chairs_mu_distances_part.npy`, which the authors **did not release**. `scripts/compute_mu_distances.py` (added in this fork) reconstructs them from the SPAGHETTI backbone + the dataset's `zh_0.npy`, then this cell persists them to Drive. Only needed for (fine)tuning, not inference.

In [ ]:
if COMPUTE_MU_DISTANCES and dataset_target.exists() and not (dataset_target / 'chairs_mu_distances.npy').exists():
    !cd {REPO_DIR} && python scripts/compute_mu_distances.py
    for name in ('chairs_mu_distances.npy', 'chairs_mu_distances_part.npy'):
        f = dataset_target / name
        if f.exists():
            shutil.copy2(f, GEN_DIR / name)
            print('persisted to Drive:', GEN_DIR / name)
elif not dataset_target.exists():
    print('training dataset not downloaded — skipping')
else:
    print('adjacency files already present')

## 9 · Smoke test — sketch → 3D mesh, end to end

Runs `run.py` on the repo's bundled `sketch.png`. This exercises the whole stack: PASTA sketch encoder (+ ISG-Net) → SPAGHETTI decode → marching cubes → `.obj` saved to `Drive/PASTA/outputs/`.

First run takes a few minutes: the 8.4 GB checkpoint streams from Drive and is loaded to **CPU RAM first** (patched in this fork). If the free-tier VM runs out of RAM here, switch to a High-RAM runtime — or run the optional *slim checkpoint* cell at the end of this notebook once, which strips the bundled optimizer state (~⅔ of the file).

In [ ]:
if RUN_SMOKE_TEST:
    # pre-flight: catches the classic "fresh VM, skipped Session restore" mistake
    required = {
        'Drive mount + section 5 downloads': pasta_ckpt,
        'SPAGHETTI backbone (section 5.2)': backbone_dir / 'model',
        'SPAGHETTI options (section 5.2)': backbone_dir / 'options.pkl',
        'assets symlinks (section 6, run EVERY session)': REPO_DIR / 'assets' / 'checkpoints' / 'occ_gmm_chairs_sym_hard' / 'options.pkl',
    }
    missing = [f'{what}: {p}' for what, p in required.items() if not Path(p).exists()]
    assert not missing, 'not ready to run — fix these first:\n  ' + '\n  '.join(missing)
    %cd {REPO_DIR}
    !python run.py --input sketch.png
    print('\noutputs on Drive:', sorted(os.listdir(OUT_DIR)))

In [ ]:
# quick 3D preview of the generated mesh
mesh_files = sorted(OUT_DIR.glob('mesh_res*.obj'))
if mesh_files:
    import matplotlib.pyplot as plt
    m = trimesh.load(mesh_files[-1], force='mesh')
    v, f = m.vertices, m.faces
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_trisurf(v[:, 0], v[:, 2], f, v[:, 1], linewidth=0.05, antialiased=True)
    ax.set_axis_off(); ax.set_box_aspect((1, 1, 1))
    plt.title(mesh_files[-1].name); plt.show()
else:
    print('no mesh outputs yet — run the smoke test above')

## 10 · Evaluation on AmateurSketch-3D (optional, manual data)

`eval.py` reproduces the paper's Chamfer-distance benchmark, but the inputs are **license-gated** and must be fetched by you once:

1. **AmateurSketch-3D (chairs)** — request/download from the [SketchX downloads page](https://sketchx.eecs.qmul.ac.uk/downloads/); it provides 3 hand-drawn sketch views per chair.
2. **ShapeNet chair meshes** — the matching `model.obj` files from [ShapeNetCore](https://shapenet.org/) (chair synset `03001627`), which requires accepting ShapeNet's terms.
3. Assemble one folder per shape and zip it as `shapenet_amateur.zip`:

```
shapenet_amateur/
  <shapenet_id>/
    1.png  2.png  3.png     # the three sketch views
    model.obj               # the ShapeNet mesh
```

4. Upload the zip to `Drive/PASTA/archives/manual/shapenet_amateur.zip`, re-run the *Session restore* cells, set `RUN_EVAL = True` in the config, and run the next cell. The LLaVA features downloaded in 5.3 are matched to these ids automatically (fork patch: path is repo-relative, override with `PASTA_LLAVA_AMATEUR`).

In [ ]:
if RUN_EVAL:
    assert eval_target.exists(), 'assets/data/shapenet_amateur is missing — follow the instructions above'
    %cd {REPO_DIR}
    !python eval.py --tag chairs --spaghetti_tag chairs_sym_hard
else:
    print('RUN_EVAL is False — skipping')

## 11 · Push run results to GitHub

Copies small artifacts (meshes, images) from this run into `colab_runs/<timestamp>/` and pushes to `main`, so a `git pull` on your laptop shows exactly what happened on Colab. Requires the `GITHUB_TOKEN` secret from step 3.

In [ ]:
if PUSH_RUN_RESULTS and token:
    import datetime, json, subprocess
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_dir = REPO_DIR / 'colab_runs' / stamp
    run_dir.mkdir(parents=True, exist_ok=True)
    for pat in ('*.obj', '*.png', '*.txt', '*.json'):
        for f in OUT_DIR.glob(pat):
            if f.stat().st_size < 20e6:
                shutil.copy2(f, run_dir / f.name)
    meta = {
        'timestamp': stamp,
        'gpu': torch.cuda.get_device_name(0),
        'torch': torch.__version__,
        'commit': subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                                 capture_output=True, text=True, cwd=REPO_DIR).stdout.strip(),
        'smoke_test': RUN_SMOKE_TEST, 'eval': RUN_EVAL,
    }
    (run_dir / 'run_info.json').write_text(json.dumps(meta, indent=2))
    %cd {REPO_DIR}
    !git add colab_runs
    !git -c user.name="Colab Runner" -c user.email="colab@users.noreply.github.com" commit -m "Colab run {stamp}" || echo "nothing new to commit"
    !git push origin HEAD:main && echo "pushed — 'git pull' locally to see this run"
elif PUSH_RUN_RESULTS:
    print('no GITHUB_TOKEN secret — cannot push. Add it (repo scope) via the key sidebar and re-run from step 3.')
else:
    print('PUSH_RUN_RESULTS is False — skipping')

## Appendix A · Optional: slim checkpoint (RAM relief)

The released checkpoint bundles optimizer + scheduler state (useful only for resuming the authors' exact training). This writes a model-only copy `model_slim` next to the original on Drive. To use it: rename `model` → `model_full` and `model_slim` → `model`. For Fusion360 finetuning you will re-initialize the optimizer anyway.

In [ ]:
MAKE_SLIM_CHECKPOINT = False
if MAKE_SLIM_CHECKPOINT:
    sd = torch.load(pasta_ckpt, map_location='cpu', mmap=True, weights_only=False)
    slim = {'model': sd['model'], 'optimizer': None, 'scheduler': None, 'epoch': sd.get('epoch')}
    torch.save(slim, pasta_ckpt.with_name('model_slim'))
    del sd, slim
    print('wrote', pasta_ckpt.with_name('model_slim'))

## Appendix B · Roadmap to Fusion360 finetuning

Everything below is now on Drive; finetuning on the **Fusion360 Gallery** dataset means reproducing the chair pipeline for a new category:

1. **SPAGHETTI backbone for Fusion360 shapes** — either finetune `occ_gmm_chairs_sym_hard` or train a SPAGHETTI on Fusion360 meshes (watertight-processed, unit-sphere normalized like ShapeNet was), producing a new `occ_gmm_<tag>` checkpoint + per-shape `zh_0.npy` embeddings.
2. **Sketches** — generate synthetic sketches per shape (SENS used NPR renders from 6 views + CLIPasso; `data_loaders/` expects `npr/view_<i>/` PNGs and `<data_tag>/svg/view_<i>/` SVGs mirroring `dataset_chair_preprocess`).
3. **LLaVA text-aligned features** — run LLaVA-7B on each sketch view and dump `[tokens, 4096]` `.npy` files; point the loaders at them via the env vars `PASTA_LLAVA_NPR` / `PASTA_LLAVA_CLIPASSO` / `PASTA_LLAVA_PROSKETCH` (fork patch).
4. **Adjacency prereqs** — rerun `scripts/compute_mu_distances.py --zh_path <fusion360 zh_0.npy>`.
5. **Finetune** — start `trainer_adj_predictor.py` from the PASTA weights: copy `sketch2spaghetti_chairs/model` to `sketch2spaghetti_<newtag>/model` and launch with `--tag <newtag>` (the trainer resumes from whatever checkpoint sits in its tag folder).

The `dataset_chair_preprocess` folder downloaded here is the authoritative reference for every file format involved.